# Materials Project Data Cleaning

This notebook loads the raw Materials Project export, keeps the fields most useful for Part 2, performs light cleaning, and saves a cleaned materials dataset for clustering.

In [1]:
# Import the libraries used in this notebook section.
from pathlib import Path
import ast

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [2]:
# Set up project paths so files can be read and outputs can be organised consistently.
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

raw_path = project_root / "data" / "raw" / "materials_project_materials.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "materials_project_materials_cleaned.csv"
clustering_output_path = processed_dir / "materials_for_clustering.csv"
clustering_reference_output_path = processed_dir / "materials_clustering_reference.csv"
clustering_features_output_path = processed_dir / "materials_clustering_features.csv"

print("Raw file:", raw_path)
print("Cleaned output file:", output_path)
print("Clustering-ready output file:", clustering_output_path)
print("Clustering reference output file:", clustering_reference_output_path)
print("Clustering features output file:", clustering_features_output_path)

Raw file: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\raw\materials_project_materials.csv
Cleaned output file: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\materials_project_materials_cleaned.csv
Clustering-ready output file: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\materials_for_clustering.csv
Clustering reference output file: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\materials_clustering_reference.csv
Clustering features output file: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\materials_clustering_features.csv


In [3]:
# Check that the expected input files are available before continuing.
if not raw_path.exists():
    raise FileNotFoundError(f"Could not find raw materials file at {raw_path}")

# Load the dataset needed for the next analysis step.
df_raw = pd.read_csv(raw_path, low_memory=False)
# Inspect the data to confirm the structure and values look reasonable.
print("Raw shape:", df_raw.shape)
display(df_raw.head())

Raw shape: (33951, 14)


,material_id,formula_pretty,elements,nelements,nsites,density,density_atomic,volume,band_gap,energy_per_atom,formation_energy_per_atom,is_stable,is_metal,symmetry
0,mp-862690,Ac,['Ac'],1,4,8.170182,46.136350,184.545401,0.0,-68.637479,0.000000,True,True,Hexagonal
1,mp-1183076,Ac2AgPb,"['Ac', 'Ag', 'Pb']",3,4,9.490476,33.640774,134.563097,0.0,-54.209271,-0.480278,True,True,Cubic
2,mp-1183086,Ac2CdHg,"['Ac', 'Cd', 'Hg']",3,4,9.570977,33.268160,133.072641,0.0,-52.120621,-0.433682,True,True,Cubic
3,mp-862786,Ac2CuGe,"['Ac', 'Cu', 'Ge']",3,4,8.598305,28.494770,113.979079,0.0,-40.884684,-0.386251,True,True,Cubic
4,mp-861883,Ac2CuIr,"['Ac', 'Cu', 'Ir']",3,4,11.322190,26.023879,104.095515,0.0,-50.462576,-0.342299,True,True,Cubic


## Inspect columns and missingness

In [4]:
# Inspect the data to confirm the structure and values look reasonable.
print(df_raw.columns.tolist())

# Handle missing values so later transformations and models do not fail.
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().sum() / len(df_raw)) * 100,
    "nunique": df_raw.nunique(dropna=False)
}).sort_values("missing_pct", ascending=False)

display(missing_summary)

['material_id', 'formula_pretty', 'elements', 'nelements', 'nsites', 'density', 'density_atomic', 'volume', 'band_gap', 'energy_per_atom', 'formation_energy_per_atom', 'is_stable', 'is_metal', 'symmetry']


,missing_count,missing_pct,nunique
material_id,0,0.0,33951
formula_pretty,0,0.0,33801
elements,0,0.0,22182
nelements,0,0.0,6
nsites,0,0.0,175
density,0,0.0,33951
density_atomic,0,0.0,33951
volume,0,0.0,33951
band_gap,0,0.0,15401
energy_per_atom,0,0.0,33949


## Keep Part 2 fields

In [5]:
keep_cols = [
    "material_id",
    "formula_pretty",
    "elements",
    "nelements",
    "nsites",
    "density",
    "density_atomic",
    "volume",
    "band_gap",
    "energy_per_atom",
    "formation_energy_per_atom",
    "is_stable",
    "is_metal",
    "symmetry",
]

# Filter the data to keep the records relevant for this step.
existing_keep_cols = [col for col in keep_cols if col in df_raw.columns]
df = df_raw[existing_keep_cols].copy()

# Inspect the data to confirm the structure and values look reasonable.
print("Kept columns:", existing_keep_cols)
print("Subset shape:", df.shape)
display(df.head())

Kept columns: ['material_id', 'formula_pretty', 'elements', 'nelements', 'nsites', 'density', 'density_atomic', 'volume', 'band_gap', 'energy_per_atom', 'formation_energy_per_atom', 'is_stable', 'is_metal', 'symmetry']
Subset shape: (33951, 14)


,material_id,formula_pretty,elements,nelements,nsites,density,density_atomic,volume,band_gap,energy_per_atom,formation_energy_per_atom,is_stable,is_metal,symmetry
0,mp-862690,Ac,['Ac'],1,4,8.170182,46.136350,184.545401,0.0,-68.637479,0.000000,True,True,Hexagonal
1,mp-1183076,Ac2AgPb,"['Ac', 'Ag', 'Pb']",3,4,9.490476,33.640774,134.563097,0.0,-54.209271,-0.480278,True,True,Cubic
2,mp-1183086,Ac2CdHg,"['Ac', 'Cd', 'Hg']",3,4,9.570977,33.268160,133.072641,0.0,-52.120621,-0.433682,True,True,Cubic
3,mp-862786,Ac2CuGe,"['Ac', 'Cu', 'Ge']",3,4,8.598305,28.494770,113.979079,0.0,-40.884684,-0.386251,True,True,Cubic
4,mp-861883,Ac2CuIr,"['Ac', 'Cu', 'Ir']",3,4,11.322190,26.023879,104.095515,0.0,-50.462576,-0.342299,True,True,Cubic


## Basic cleaning

In [6]:
def parse_elements(value):
    # Handle missing values so later transformations and models do not fail.
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(x) for x in parsed]
        except (ValueError, SyntaxError):
            pass
    return [text]

if "formula_pretty" in df.columns:
    # Convert values into analysis-friendly numeric, date, or text formats.
    df["formula_pretty"] = df["formula_pretty"].astype(str).str.strip()
    # Filter the data to keep the records relevant for this step.
    df["formula_match_key"] = df["formula_pretty"].str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")

if "elements" in df.columns:
    df["elements_list"] = df["elements"].apply(parse_elements)
    # Combine related tables so each record has the fields needed for analysis.
    df["material_family_key"] = df["elements_list"].apply(lambda x: "_".join(sorted(x)) if x else pd.NA)
    df["material_family_match_key"] = df["material_family_key"].astype(str).str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")

numeric_cols = [
    "nelements",
    "nsites",
    "density",
    "density_atomic",
    "volume",
    "band_gap",
    "energy_per_atom",
    "formation_energy_per_atom",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.drop_duplicates(subset=[col for col in ["material_id", "formula_pretty"] if col in df.columns])

# Inspect the data to confirm the structure and values look reasonable.
print("Cleaned shape:", df.shape)
display(df.head())

Cleaned shape: (33951, 18)


,material_id,formula_pretty,elements,nelements,nsites,density,density_atomic,volume,band_gap,energy_per_atom,formation_energy_per_atom,is_stable,is_metal,symmetry,formula_match_key,elements_list,material_family_key,material_family_match_key
0,mp-862690,Ac,['Ac'],1,4,8.170182,46.136350,184.545401,0.0,-68.637479,0.000000,True,True,Hexagonal,ac,[Ac],Ac,ac
1,mp-1183076,Ac2AgPb,"['Ac', 'Ag', 'Pb']",3,4,9.490476,33.640774,134.563097,0.0,-54.209271,-0.480278,True,True,Cubic,ac2agpb,"[Ac, Ag, Pb]",Ac_Ag_Pb,ac_ag_pb
2,mp-1183086,Ac2CdHg,"['Ac', 'Cd', 'Hg']",3,4,9.570977,33.268160,133.072641,0.0,-52.120621,-0.433682,True,True,Cubic,ac2cdhg,"[Ac, Cd, Hg]",Ac_Cd_Hg,ac_cd_hg
3,mp-862786,Ac2CuGe,"['Ac', 'Cu', 'Ge']",3,4,8.598305,28.494770,113.979079,0.0,-40.884684,-0.386251,True,True,Cubic,ac2cuge,"[Ac, Cu, Ge]",Ac_Cu_Ge,ac_cu_ge
4,mp-861883,Ac2CuIr,"['Ac', 'Cu', 'Ir']",3,4,11.322190,26.023879,104.095515,0.0,-50.462576,-0.342299,True,True,Cubic,ac2cuir,"[Ac, Cu, Ir]",Ac_Cu_Ir,ac_cu_ir


## Filter to procurement-relevant clustering candidates

In [7]:
# Choose the columns that will be used as features or summary variables.
clustering_feature_cols = [
    "nelements",
    "nsites",
    "density",
    "density_atomic",
    "volume",
    "band_gap",
    "energy_per_atom",
    "formation_energy_per_atom",
]

# Filter the data to keep the records relevant for this step.
existing_feature_cols = [col for col in clustering_feature_cols if col in df.columns]

df_cluster = df.copy()

# Keep stable materials where the field exists
if "is_stable" in df_cluster.columns:
    df_cluster = df_cluster[df_cluster["is_stable"] == True].copy()

# Restrict to simpler compositions that are more likely to be practical and easier to match to cost data
if "nelements" in df_cluster.columns:
    df_cluster = df_cluster[df_cluster["nelements"].between(1, 6, inclusive="both")].copy()

# Remove rows with too many missing core clustering features
if existing_feature_cols:
    # Handle missing values so later transformations and models do not fail.
    df_cluster["core_feature_missing_pct"] = df_cluster[existing_feature_cols].isna().mean(axis=1)
    df_cluster = df_cluster[df_cluster["core_feature_missing_pct"] <= 0.40].copy()

# Require at least a minimum number of present clustering features
if existing_feature_cols:
    df_cluster["core_feature_count"] = df_cluster[existing_feature_cols].notna().sum(axis=1)
    min_feature_count = min(4, len(existing_feature_cols))
    df_cluster = df_cluster[df_cluster["core_feature_count"] >= min_feature_count].copy()

# Inspect the data to confirm the structure and values look reasonable.
print("Clustering-ready shape:", df_cluster.shape)
display(df_cluster.head())

Clustering-ready shape: (33951, 20)


,material_id,formula_pretty,elements,nelements,nsites,density,density_atomic,volume,band_gap,energy_per_atom,formation_energy_per_atom,is_stable,is_metal,symmetry,formula_match_key,elements_list,material_family_key,material_family_match_key,core_feature_missing_pct,core_feature_count
0,mp-862690,Ac,['Ac'],1,4,8.170182,46.136350,184.545401,0.0,-68.637479,0.000000,True,True,Hexagonal,ac,[Ac],Ac,ac,0.0,8
1,mp-1183076,Ac2AgPb,"['Ac', 'Ag', 'Pb']",3,4,9.490476,33.640774,134.563097,0.0,-54.209271,-0.480278,True,True,Cubic,ac2agpb,"[Ac, Ag, Pb]",Ac_Ag_Pb,ac_ag_pb,0.0,8
2,mp-1183086,Ac2CdHg,"['Ac', 'Cd', 'Hg']",3,4,9.570977,33.268160,133.072641,0.0,-52.120621,-0.433682,True,True,Cubic,ac2cdhg,"[Ac, Cd, Hg]",Ac_Cd_Hg,ac_cd_hg,0.0,8
3,mp-862786,Ac2CuGe,"['Ac', 'Cu', 'Ge']",3,4,8.598305,28.494770,113.979079,0.0,-40.884684,-0.386251,True,True,Cubic,ac2cuge,"[Ac, Cu, Ge]",Ac_Cu_Ge,ac_cu_ge,0.0,8
4,mp-861883,Ac2CuIr,"['Ac', 'Cu', 'Ir']",3,4,11.322190,26.023879,104.095515,0.0,-50.462576,-0.342299,True,True,Cubic,ac2cuir,"[Ac, Cu, Ir]",Ac_Cu_Ir,ac_cu_ir,0.0,8


## Build final clustering reference and feature tables

In [8]:
reference_cols = [
    "material_id",
    "formula_pretty",
    "formula_match_key",
    "elements",
    "elements_list",
    "material_family_key",
    "material_family_match_key",
    "is_stable",
    "is_metal",
    "symmetry",
]

# Filter the data to keep the records relevant for this step.
existing_reference_cols = [col for col in reference_cols if col in df_cluster.columns]
# Group similar records and evaluate the quality of the cluster structure.
df_cluster_reference = df_cluster[existing_reference_cols].copy()

# Choose the columns that will be used as features or summary variables.
df_cluster_features = df_cluster[existing_feature_cols].copy()

if "is_metal" in df_cluster.columns:
    # Convert values into analysis-friendly numeric, date, or text formats.
    df_cluster_features["is_metal_flag"] = df_cluster["is_metal"].astype(float)

for col in df_cluster_features.columns:
    df_cluster_features[col] = pd.to_numeric(df_cluster_features[col], errors="coerce")
    # Handle missing values so later transformations and models do not fail.
    if df_cluster_features[col].isna().any():
        df_cluster_features[col] = df_cluster_features[col].fillna(df_cluster_features[col].median())

# Inspect the data to confirm the structure and values look reasonable.
print("Reference table shape:", df_cluster_reference.shape)
display(df_cluster_reference.head())

print("Feature matrix shape:", df_cluster_features.shape)
display(df_cluster_features.head())

Reference table shape: (33951, 10)


,material_id,formula_pretty,formula_match_key,elements,elements_list,material_family_key,material_family_match_key,is_stable,is_metal,symmetry
0,mp-862690,Ac,ac,['Ac'],[Ac],Ac,ac,True,True,Hexagonal
1,mp-1183076,Ac2AgPb,ac2agpb,"['Ac', 'Ag', 'Pb']","[Ac, Ag, Pb]",Ac_Ag_Pb,ac_ag_pb,True,True,Cubic
2,mp-1183086,Ac2CdHg,ac2cdhg,"['Ac', 'Cd', 'Hg']","[Ac, Cd, Hg]",Ac_Cd_Hg,ac_cd_hg,True,True,Cubic
3,mp-862786,Ac2CuGe,ac2cuge,"['Ac', 'Cu', 'Ge']","[Ac, Cu, Ge]",Ac_Cu_Ge,ac_cu_ge,True,True,Cubic
4,mp-861883,Ac2CuIr,ac2cuir,"['Ac', 'Cu', 'Ir']","[Ac, Cu, Ir]",Ac_Cu_Ir,ac_cu_ir,True,True,Cubic


Feature matrix shape: (33951, 9)


,nelements,nsites,density,density_atomic,volume,band_gap,energy_per_atom,formation_energy_per_atom,is_metal_flag
0,1,4,8.170182,46.136350,184.545401,0.0,-68.637479,0.000000,1.0
1,3,4,9.490476,33.640774,134.563097,0.0,-54.209271,-0.480278,1.0
2,3,4,9.570977,33.268160,133.072641,0.0,-52.120621,-0.433682,1.0
3,3,4,8.598305,28.494770,113.979079,0.0,-40.884684,-0.386251,1.0
4,3,4,11.322190,26.023879,104.095515,0.0,-50.462576,-0.342299,1.0


## Save cleaned outputs

In [9]:
# Save the processed output so later notebooks or report sections can reuse it.
df.to_csv(output_path, index=False)
df_cluster.to_csv(clustering_output_path, index=False)
# Group similar records and evaluate the quality of the cluster structure.
df_cluster_reference.to_csv(clustering_reference_output_path, index=False)
df_cluster_features.to_csv(clustering_features_output_path, index=False)

print(f"Saved cleaned materials dataset to: {output_path}")
print(f"Saved clustering-ready materials dataset to: {clustering_output_path}")
print(f"Saved clustering reference table to: {clustering_reference_output_path}")
print(f"Saved clustering feature matrix to: {clustering_features_output_path}")

Saved cleaned materials dataset to: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\materials_project_materials_cleaned.csv
Saved clustering-ready materials dataset to: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\materials_for_clustering.csv
Saved clustering reference table to: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\materials_clustering_reference.csv
Saved clustering feature matrix to: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\materials_clustering_features.csv
